# `pseudonimizzatore_legale` playground

This notebook is a practical tour of the public API. All names and identifiers below are invented. The library works locally; optional NER may download model files on first use, but document text is not sent to a hosted inference service.

> **Important:** this is pseudonymization, not a guarantee of anonymity. Always review sensitive output before publication.

In [1]:
from pathlib import Path
import sys

# Work both from the repository root and from its notebooks/ directory.
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pseudonimizzatore_legale import Config, __version__, anonymize, anonymize_batch

print(f"pseudonimizzatore-legale {__version__}")

pseudonimizzatore-legale 0.2.0


## 1. Start with one string

`anonymize()` returns the pseudonymized text and an audit report.

In [2]:
source = (
    "Il ricorrente Mario Rossi, C.F. RSSMRA80A01H501U, "
    "è rappresentato dall'avv. Laura Bianchi."
)

output, report = anonymize(source)
print(output)

Il ricorrente Ricorrente_1, C.F. CF_1, è rappresentato dall'avv. Difensore_1.


The party, tax code, and lawyer receive readable tags. Tags are stable only within this document, which avoids creating a cross-document identifier.

In [3]:
print("status:", report.status)
print("entities:", report.entities)
print("mapping:")
for original, tag in report.mapping.items():
    print(f"  {original!r} -> {tag}")

status: passed_checks
entities: 3
mapping:
  'Mario Rossi' -> Ricorrente_1
  'RSSMRA80A01H501U' -> CF_1
  'Laura Bianchi' -> Difensore_1


## 2. Inspect why text was changed

`decisions` records the accepted redactions and protected spans. Its offsets refer to normalized input; residual findings use output offsets.

In [4]:
for item in report.decisions:
    replacement = f" -> {item.replacement}" if item.replacement else ""
    print(
        f"{item.action:6s} {item.kind:12s} {item.text!r}{replacement}"
        f"  evidence: {item.source}; reason: {item.reason}"
    )

redact PERSON       'Mario Rossi' -> Ricorrente_1  evidence: regex:biografico+regex:party_role; reason: resolved full-name alias
redact CF           'RSSMRA80A01H501U' -> CF_1  evidence: regex:cf; reason: structured identifier
redact PERSON       'Laura Bianchi' -> Difensore_1  evidence: regex:counsel+regex:counsel_list+regex:difeso_da; reason: resolved full-name alias


## 3. Change policy explicitly

Private companies are kept by default. Set `companies=True` when your use case requires replacing them.

In [5]:
company_text = (
    "La Alfa Costruzioni S.r.l. ricorre, rappresentata "
    "dall'avv. Laura Bianchi."
)

default_output, _ = anonymize(company_text)
company_output, _ = anonymize(company_text, Config(companies=True))

print("default:         ", default_output)
print("companies=True: ", company_output)

default:          La Alfa Costruzioni S.r.l. ricorre, rappresentata dall'avv. Difensore_1.
companies=True:  Società_1 ricorre, rappresentata dall'avv. Difensore_1.


## 4. Keep one identity consistent

Legal decisions often write the same person surname-first in the heading, in ordinary order in prose, and later by surname alone. Once strong context establishes the identity, the library propagates likely local aliases.

In [6]:
decision = """sul ricorso proposto da
BENATTI ROSSELLA
-ricorrente-
Rossella Benatti insiste. La Benatti ricorre.
"""

consistent_output, consistent_report = anonymize(decision)
print(consistent_output)
print(consistent_report.mapping)

sul ricorso proposto da
Ricorrente_1
-ricorrente-
Ricorrente_1 insiste. La Ricorrente_1 ricorre.

{'BENATTI ROSSELLA': 'Ricorrente_1'}


## 5. Treat verification as a review queue

The primary regex detector deliberately requires legal evidence before changing a name-like phrase. A broader, non-mutating verifier can still flag an uncued candidate for human review.

In [7]:
uncued = "La decisione riguarda Luca Romano senza altre indicazioni."
uncued_output, uncued_report = anonymize(uncued)

print(uncued_output)
print("status:", uncued_report.status)
for finding in uncued_report.residuals:
    print(f"  review {finding.text!r}: {finding.reason}")

La decisione riguarda Luca Romano senza altre indicazioni.
status: needs_review
  review 'Luca Romano': unresolved capitalized name candidate


A `passed_checks` result means only that these inexpensive checks found nothing. It is not publication clearance.

## 6. Optional transformer NER

Install `.[ner]` first. NER adds person mentions to the same identity and policy pipeline. The first run may download model weights from Hugging Face. Set `RUN_NER = True` only when you want that behavior.

In [8]:
RUN_NER = False
NER_MODEL = "DeepMount00/Italian_NER_XXL_v2"

if RUN_NER:
    ner_output, ner_report = anonymize(
        uncued, ner=NER_MODEL, ner_threshold=0.3, ner_device="cpu"
    )
    print(ner_output)
    print(ner_report.mapping)
else:
    print("NER example skipped. Set RUN_NER = True to run it.")

NER example skipped. Set RUN_NER = True to run it.


On the included corpus, regex-only complete-entity recall is 86.8%; Italian NER XXL raises it to 91.8%. The recall-first trade-off is lower precision and some additional removal of uncued public-role aliases.

## 7. Process files in a directory

Batch processing preserves the input directory layout. Sidecars are optional and sensitive because they include original-to-tag mappings. This example leaves sidecars disabled.

In [9]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temporary:
    workspace = Path(temporary)
    input_dir = workspace / "input"
    output_dir = workspace / "output"
    input_dir.mkdir()
    (input_dir / "decisione.txt").write_text(source, encoding="utf-8")

    summary = anonymize_batch(input_dir, output_dir, workers=1)
    print((output_dir / "decisione.txt").read_text(encoding="utf-8"))
    print({key: summary[key] for key in ("files", "errors", "entities", "needs_review")})

Il ricorrente Ricorrente_1, C.F. CF_1, è rappresentato dall'avv. Difensore_1.
{'files': 1, 'errors': 0, 'entities': 3, 'needs_review': 0}


## 8. Try your own synthetic text

Edit the value below. Avoid pasting real client data into a notebook that might later be committed or shared.

In [10]:
your_text = "Il sig. Giovanni Esposito, nato il 04/05/1981, propone ricorso."
your_output, your_report = anonymize(your_text)
print(your_output)
print("status:", your_report.status)
print("mapping:", your_report.mapping)

Il sig. Nominativo_1, nato il Data_nascita_1, propone ricorso.
status: passed_checks
mapping: {'Giovanni Esposito': 'Nominativo_1', '04/05/1981': 'Data_nascita_1'}


## Where to go next

- Read `README.md` for installation, API choices, and batch safety.
- Read `ARCHITECTURE.md` to understand detectors, identity resolution, late policy, and verification.
- Run `python evaluation/score_corpus.py` for the full 103-document benchmark.
- Run `python -m pytest -q` before changing detection or policy behavior.